# image-ai Colab execution notebook

## Specification (Colab-first)
- Verification environment: **Google Colab**
- Entry point: **this notebook (`notebooks/run.ipynb`)**
- Execution order: environment check -> repository clone -> dependency install -> generation run -> output check
- Repository source is fixed to `https://github.com/koba-84/image.git` (branch `chore/autonomous-pr-guardrails`) and cloned to `/content/image`.


In [ ]:
!python --version
!nvidia-smi || echo "GPU not available on this runtime."

In [ ]:
ROOT = '/content/image'
REPO_URL = 'https://github.com/koba-84/image.git'
REPO_BRANCH = 'chore/autonomous-pr-guardrails'

%cd /content
!rm -rf /content/image && git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} /content/image
!test -f /content/image/pyproject.toml
print(f'Repository root: {ROOT} (branch: {REPO_BRANCH})')


: 

In [ ]:
%cd /content/image
!command -v uv >/dev/null || (curl -LsSf https://astral.sh/uv/install.sh | sh)
!export PATH="$HOME/.local/bin:$PATH" && uv sync --project /content/image --python 3.12

: 

In [ ]:
!cd /content/image && export PATH="$HOME/.local/bin:$PATH" && PYTHONPATH=/content/image uv run --python 3.12 -- python - <<'PY'
from pathlib import Path
import torch
from omegaconf import OmegaConf
from image_ai.pipeline_runner import load_pipeline_for_mode

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model_files = sorted(Path('image_ai/conf/model').glob('*.yaml'))
if not model_files:
    raise RuntimeError('No model config files found in image_ai/conf/model')

loaded = []
for cfg_path in model_files:
    cfg = OmegaConf.to_container(OmegaConf.load(cfg_path), resolve=True)
    model_variant = cfg.get('model_variant')
    model_id = cfg.get('model_id')
    if not model_id:
        raise RuntimeError(f'model_id missing in {cfg_path}')

    mode = 'inpaint' if model_variant == 'sd_inpaint' else 'text2img'
    pipe = load_pipeline_for_mode(mode, model_id, dtype)
    if pipe is None:
        raise RuntimeError(f'failed to load: {model_variant} ({model_id})')

    loaded.append((model_variant, mode, model_id))
    print(f'OK: {model_variant} [{mode}] -> {model_id}')

print(f'Loaded {len(loaded)} model presets successfully.')
PY

: 

In [ ]:
!cd /content/image && export PATH="$HOME/.local/bin:$PATH" && uv run --project /content/image --python 3.12 -m image_ai.cli --mode text2img \
  --model-id stabilityai/stable-diffusion-xl-base-1.0 \
  --prompt "cinematic night city, ultra detailed" \
  --output /content/image/outputs/colab_text2img.png \
  --steps 24 --guidance-scale 6.0 \
  --height 768 --width 768

: 

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

out = Path('/content/image/outputs/colab_text2img.png')
if out.exists():
    display(Image.open(out))
    print(f'Generated: {out}')
else:
    print('Output image not found. Check the previous cell logs.')

: 